# SIIM-ISIC 2020 — Final Full Same-Patient Leakage Audit

**Standalone one-run notebook for same-patient exposure audit**

This notebook starts from the **raw SIIM-ISIC 2020 training metadata + JPEG/PNG images only**.
It does **not** use the previous ISIC notebook, its output, old model checkpoints, or old predictions.

## Final experimental question

For fixed dermoscopic test images, what changes when training is deliberately allowed to contain
**other images from the same test patients**, while direct test-image reuse and major metadata
composition differences are excluded?

The final design therefore enforces:

- identical fixed test images for rigorous and controlled-leak arms;
- zero exact test-image overlap in both arms;
- zero decoded-pixel duplicate overlap in both arms;
- zero test-patient overlap in the rigorous training arm;
- an identical patient-disjoint validation set for both arms;
- exact matching of controlled-leak replacements on:
  - binary target,
  - diagnosis,
  - general anatomical site,
  - sex;
- nearest-age matching within each exact stratum;
- equal training-set size and identical matched-stratum counts;
- strict deterministic execution;
- patient-cluster bootstrap confidence intervals and patient-cluster paired randomization;
- validation-only operating-threshold selection for thresholded metrics;
- optional `lesion_id` integration from official metadata v2;
- same-lesion vs different-lesion exposure analysis when `lesion_id` is available;
- corrected quantitative Grad-CAM from the final ResNet-50 models.

## Lesion metadata

The official ISIC 2020 metadata v2 includes `lesion_id`. The notebook:

1. prefers a local metadata-v2 CSV when attached;
2. otherwise attempts to download the official metadata-v2 CSV when Internet is available;
3. if neither is available, continues safely without lesion-level analysis.

The primary leakage experiment itself does **not** depend on `lesion_id`.

## XAI

The final XAI analysis is deliberately conservative:

- ResNet-50 only;
- a seeded, prediction-independent OOF subset;
- identical true-class target for both paired models;
- paired Grad-CAM cosine similarity;
- paired Grad-CAM Pearson correlation;
- normalized attribution-centroid displacement;
- patient-cluster uncertainty;
- exposure-stratified attribution analysis.

This is treated as **secondary attribution-stability evidence**, not proof of a causal memorization mechanism.

## References used for the final design

- Rotemberg et al., *Scientific Data* 8, 34 (2021), patient-centric ISIC 2020 dataset.
- Official SIIM-ISIC 2020 challenge dataset page and metadata v2.
- CLAIM 2024 reporting guidance for patient-level or higher disjointness in medical imaging.
- scikit-learn `StratifiedGroupKFold`.

**Interpretation rule:** no assertion requires leakage to improve performance. Positive, null, heterogeneous,
or negative corrected effects are all scientifically valid outcomes.

In [1]:
# =========================
# 1. Strict deterministic environment + configuration
# =========================
import os
os.environ["PYTHONHASHSEED"] = "42"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

from pathlib import Path
import re, sys, json, math, time, random, hashlib, zipfile, warnings, platform, urllib.request
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as T
import torchvision.models as tvm

from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score, f1_score,
    balanced_accuracy_score, matthews_corrcoef, recall_score,
    brier_score_loss, log_loss, roc_curve
)
from scipy.stats import spearmanr

SEED = 42
FOLDS = 5
VAL_FOLDS = 5

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0
EPOCHS = 12
PATIENCE = 3
LR = 1e-4
WEIGHT_DECAY = 1e-4
PRETRAINED = True

PRIMARY_PER_CLASS = 584
NEGATIVE_CANDIDATE_MULTIPLIER = 3
LEAK_FRACTION = 1.0

RUN_ARCHITECTURES = ["ResNet50", "ConvNeXtTiny"]

# Retains the previous low-cost secondary imbalance analysis, but it is explicitly
# described as a capped-imbalance sensitivity experiment, not native prevalence.
RUN_CAPPED_IMBALANCE_SENSITIVITY = True
IMBALANCE_MAX_BENIGN = 3000
IMBALANCE_MODEL = "ResNet50"

BOOTSTRAPS = 2000
PERMUTATIONS = 2000

# Final corrected XAI: ResNet-50 only, fixed OOF subset.
XAI_ARCHITECTURE = "ResNet50"
XAI_MAX_PER_FOLD = 100
XAI_VISUAL_TOTAL = 16

# Metadata v2 is useful for lesion_id. Failure to obtain it does not stop the run.
AUTO_FETCH_METADATA_V2 = True
METADATA_V2_URL = (
    "https://isic-challenge-data.s3.amazonaws.com/2020/"
    "ISIC_2020_Training_GroundTruth_v2.csv"
)

# Optional manual overrides; normally leave blank on Kaggle.
METADATA_CSV_OVERRIDE = ""
IMAGE_ROOT_OVERRIDE = ""
METADATA_V2_OVERRIDE = ""

BASE = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/mnt/data")
OUT = BASE / "isic_final_full_audit"
SPLIT_DIR = OUT / "splits"
PRED_DIR = OUT / "predictions"
XAI_DIR = OUT / "xai"
FIG_DIR = OUT / "figures"
for p in [OUT, SPLIT_DIR, PRED_DIR, XAI_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    if hasattr(torch.backends, "cuda") and hasattr(torch.backends.cuda, "matmul"):
        torch.backends.cuda.matmul.allow_tf32 = False
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.allow_tf32 = False

seed_everything()
torch.use_deterministic_algorithms(True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: CPU execution will be very slow. Use a Kaggle/Colab GPU.")

versions = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "pandas": pd.__version__,
    "cuda": torch.version.cuda,
    "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "cudnn_deterministic": bool(torch.backends.cudnn.deterministic),
    "cudnn_benchmark": bool(torch.backends.cudnn.benchmark),
    "deterministic_algorithms": bool(torch.are_deterministic_algorithms_enabled()),
    "CUBLAS_WORKSPACE_CONFIG": os.environ.get("CUBLAS_WORKSPACE_CONFIG"),
}
print(json.dumps(versions, indent=2))

Device: cuda
GPU: Tesla T4
{
  "python": "3.13.15",
  "torch": "2.11.0+cu128",
  "torchvision": "0.26.0+cu128",
  "pandas": "2.3.3",
  "cuda": "12.8",
  "device": "Tesla T4",
  "cudnn_deterministic": true,
  "cudnn_benchmark": false,
  "deterministic_algorithms": true,
  "CUBLAS_WORKSPACE_CONFIG": ":4096:8"
}


In [2]:
# =========================
# 2. Robust ISIC 2020 metadata discovery + optional official lesion_id metadata v2
# =========================
REQUIRED = {"image_name", "patient_id", "target"}
IMG_EXTS = {".jpg", ".jpeg", ".png"}

def csv_header(path):
    try:
        return set(pd.read_csv(path, nrows=2).columns)
    except Exception:
        return set()

def approx_rows(path):
    try:
        with open(path, "rb") as f:
            return max(sum(1 for _ in f) - 1, 0)
    except Exception:
        return 0

def find_main_metadata():
    if METADATA_CSV_OVERRIDE:
        p = Path(METADATA_CSV_OVERRIDE)
        if not p.exists():
            raise FileNotFoundError(p)
        if not REQUIRED.issubset(csv_header(p)):
            raise RuntimeError(f"Override metadata lacks {REQUIRED}: {p}")
        return p

    candidates = []
    for base in [Path("/kaggle/input"), Path("/content"), Path.cwd()]:
        if not base.exists():
            continue
        for p in base.rglob("*.csv"):
            cols = csv_header(p)
            if not REQUIRED.issubset(cols):
                continue
            name = p.name.lower()
            text = str(p).lower()
            n = approx_rows(p)
            score = 0
            if "groundtruth_v2" in name: score += 120
            if "train.csv" == name: score += 80
            if "train" in name: score += 30
            if "isic" in text or "melanoma" in text: score += 30
            if "lesion_id" in cols: score += 50
            if 30000 <= n <= 40000: score += 50
            candidates.append((score, n, p))

    if not candidates:
        raise FileNotFoundError(
            "Could not find SIIM-ISIC 2020 training metadata. Attach a CSV containing "
            "image_name, patient_id and target."
        )

    candidates.sort(key=lambda x: (-x[0], -x[1], len(str(x[2]))))
    print("Main metadata candidates:")
    for s, n, p in candidates[:10]:
        print(f"  score={s:3d} rows~{n:6d} {p}")
    return candidates[0][2]

MAIN_CSV = find_main_metadata()
print("Chosen main metadata:", MAIN_CSV)

def find_local_metadata_v2():
    if METADATA_V2_OVERRIDE:
        p = Path(METADATA_V2_OVERRIDE)
        if not p.exists():
            raise FileNotFoundError(p)
        if "lesion_id" not in csv_header(p):
            raise RuntimeError(f"METADATA_V2_OVERRIDE has no lesion_id: {p}")
        return p

    candidates = []
    for base in [Path("/kaggle/input"), Path("/content"), Path.cwd()]:
        if not base.exists():
            continue
        for p in base.rglob("*.csv"):
            cols = csv_header(p)
            if "image_name" in cols and "lesion_id" in cols:
                name = p.name.lower()
                score = 0
                if "groundtruth_v2" in name: score += 100
                if "metadata" in name or "train" in name: score += 20
                n = approx_rows(p)
                if 30000 <= n <= 40000: score += 50
                candidates.append((score, n, p))

    if not candidates:
        return None
    candidates.sort(key=lambda x: (-x[0], -x[1], len(str(x[2]))))
    return candidates[0][2]

META_V2 = find_local_metadata_v2()

if META_V2 is None and AUTO_FETCH_METADATA_V2:
    fetch_path = BASE / "ISIC_2020_Training_GroundTruth_v2.csv"
    try:
        print("Local lesion_id metadata v2 not found; attempting official download...")
        urllib.request.urlretrieve(METADATA_V2_URL, fetch_path)
        if "lesion_id" in csv_header(fetch_path):
            META_V2 = fetch_path
            print("Downloaded official metadata v2:", META_V2)
        else:
            print("Downloaded file did not expose lesion_id; lesion analysis will be skipped.")
    except Exception as e:
        print("Metadata v2 download unavailable; continuing without lesion_id.")
        print("  Reason:", repr(e))

main = pd.read_csv(MAIN_CSV)
main = main.loc[:, ~main.columns.duplicated()].copy()

if META_V2 is not None:
    v2 = pd.read_csv(META_V2)
    v2 = v2.loc[:, ~v2.columns.duplicated()].copy()
    lesion_cols = [
        c for c in [
            "image_name", "lesion_id", "patient_id", "sex", "age_approx",
            "anatom_site_general_challenge", "diagnosis", "benign_malignant", "target"
        ] if c in v2.columns
    ]
    v2 = v2[lesion_cols].drop_duplicates("image_name")
    # Keep values already in the main CSV; only supplement missing columns/lesion_id.
    for c in lesion_cols:
        if c == "image_name":
            continue
        if c not in main.columns:
            main = main.merge(v2[["image_name", c]], on="image_name", how="left")
        elif c == "lesion_id":
            aux = v2[["image_name", c]].rename(columns={c: "_lesion_id_v2"})
            main = main.merge(aux, on="image_name", how="left")
            main["lesion_id"] = main["lesion_id"].where(
                main["lesion_id"].notna(), main["_lesion_id_v2"]
            )
            main = main.drop(columns=["_lesion_id_v2"])
    LESION_ID_AVAILABLE = "lesion_id" in main.columns and main["lesion_id"].notna().any()
else:
    LESION_ID_AVAILABLE = "lesion_id" in main.columns and main["lesion_id"].notna().any()

print("Lesion ID available:", LESION_ID_AVAILABLE)

Main metadata candidates:
  score=160 rows~ 33126 /kaggle/input/datasets/sumaiyabinteshahid/isic-challenge-dataset-2020/ISIC_2020_Dataset/ISIC_2020_Train_Metadata.csv
Chosen main metadata: /kaggle/input/datasets/sumaiyabinteshahid/isic-challenge-dataset-2020/ISIC_2020_Dataset/ISIC_2020_Train_Metadata.csv
Lesion ID available: True


In [3]:
# =========================
# 3. Image discovery + metadata normalization
# =========================
def find_image_root(metadata_path):
    if IMAGE_ROOT_OVERRIDE:
        p = Path(IMAGE_ROOT_OVERRIDE)
        if not p.exists():
            raise FileNotFoundError(p)
        return p

    probe = main["image_name"].dropna().astype(str).head(50).tolist()
    search_roots = [metadata_path.parent, metadata_path.parent.parent]
    if Path("/kaggle/input").exists():
        search_roots.append(Path("/kaggle/input"))

    best = None
    seen = set()

    for root in search_roots:
        if not root.exists():
            continue
        key = str(root.resolve())
        if key in seen:
            continue
        seen.add(key)

        try:
            files = [
                p for p in root.rglob("*")
                if p.is_file() and p.suffix.lower() in IMG_EXTS
            ]
        except Exception:
            continue

        if not files:
            continue

        stems = {p.stem for p in files}
        frac = sum(x in stems for x in probe) / max(len(probe), 1)
        print(f"Image-root candidate: {root} | files={len(files):,} | probe={frac:.1%}")

        if best is None or frac > best[0]:
            best = (frac, root, files)
        if frac >= 0.95:
            break

    if best is None or best[0] < 0.80:
        raise FileNotFoundError(
            "Could not locate JPEG/PNG images matching SIIM-ISIC training metadata. "
            "Set IMAGE_ROOT_OVERRIDE if needed."
        )
    return best[1], best[2]

IMAGE_ROOT, image_files = find_image_root(MAIN_CSV)
print("Chosen image root:", IMAGE_ROOT)

image_index = {}
for p in sorted(image_files, key=lambda x: (len(str(x)), str(x))):
    image_index.setdefault(p.stem, str(p))

def norm_text(s, unknown="unknown"):
    if pd.isna(s):
        return unknown
    s = str(s).strip().lower()
    return s if s else unknown

meta = main.copy()
meta = meta[meta["target"].isin([0, 1])].copy()
meta = meta.dropna(subset=["image_name", "patient_id", "target"]).reset_index(drop=True)

meta["image_name"] = meta["image_name"].astype(str)
meta["Patient_ID"] = meta["patient_id"].astype(str)
meta["Label"] = meta["target"].astype(int)
meta["filepath"] = meta["image_name"].map(image_index)

missing = int(meta.filepath.isna().sum())
if missing:
    print(f"WARNING: {missing} metadata rows have no matched image and will be excluded.")
meta = meta.dropna(subset=["filepath"]).reset_index(drop=True)

meta["SexNorm"] = (
    meta["sex"].map(norm_text) if "sex" in meta.columns
    else "unknown"
)
meta["SiteNorm"] = (
    meta["anatom_site_general_challenge"].map(norm_text)
    if "anatom_site_general_challenge" in meta.columns else "unknown"
)
meta["DiagnosisNorm"] = (
    meta["diagnosis"].map(norm_text) if "diagnosis" in meta.columns
    else "unknown"
)

if "age_approx" in meta.columns:
    meta["AgeApprox"] = pd.to_numeric(meta["age_approx"], errors="coerce")
else:
    meta["AgeApprox"] = np.nan

if LESION_ID_AVAILABLE:
    meta["Lesion_ID"] = meta["lesion_id"].astype("string")
    meta.loc[meta["lesion_id"].isna(), "Lesion_ID"] = pd.NA
else:
    meta["Lesion_ID"] = pd.Series([pd.NA] * len(meta), dtype="string")

# Exact hard-matching stratum for controlled-leak replacement.
meta["MatchStratum"] = (
    "Y=" + meta.Label.astype(str)
    + "||DIAG=" + meta.DiagnosisNorm
    + "||SITE=" + meta.SiteNorm
    + "||SEX=" + meta.SexNorm
)

print(f"Usable rows: {len(meta):,}")
print(f"Patients: {meta.Patient_ID.nunique():,}")
print("Class counts:", meta.Label.value_counts().sort_index().to_dict())
print(f"Observed melanoma prevalence: {meta.Label.mean():.4%}")
print("Lesion ID coverage:", f"{100*meta.Lesion_ID.notna().mean():.1f}%")

Image-root candidate: /kaggle/input/datasets/sumaiyabinteshahid/isic-challenge-dataset-2020/ISIC_2020_Dataset | files=44,108 | probe=100.0%
Chosen image root: /kaggle/input/datasets/sumaiyabinteshahid/isic-challenge-dataset-2020/ISIC_2020_Dataset
Usable rows: 33,126
Patients: 2,056
Class counts: {0: 32542, 1: 584}
Observed melanoma prevalence: 1.7630%
Lesion ID coverage: 100.0%


In [4]:
# =========================
# 4. Full decoded-pixel fingerprinting + deterministic duplicate-safe cohorts
# =========================
FP_CACHE = OUT / "fingerprint_cache.csv"

def raw_sha256(path, chunk=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        while True:
            b = fh.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def decoded_fingerprint(path):
    # Exact decoded-pixel hash: protects against duplicate images stored with different JPEG bytes.
    with Image.open(path) as im:
        im = im.convert("RGB")
        size = im.size
        arr = np.asarray(im, dtype=np.uint8)
    h = hashlib.sha256()
    h.update(str(size).encode("utf-8"))
    h.update(arr.tobytes())
    return h.hexdigest()

_fp_cache = {}
if FP_CACHE.exists():
    try:
        q = pd.read_csv(FP_CACHE)
        if {"image_name", "raw_sha256", "decoded_fp"}.issubset(q.columns):
            _fp_cache = q.set_index("image_name")[["raw_sha256", "decoded_fp"]].to_dict("index")
    except Exception:
        _fp_cache = {}

def fingerprint_rows(df):
    rows = []
    changed = False

    for _, r in df.iterrows():
        key = str(r.image_name)
        rec = _fp_cache.get(key)
        if rec is None:
            rec = {
                "raw_sha256": raw_sha256(r.filepath),
                "decoded_fp": decoded_fingerprint(r.filepath),
            }
            _fp_cache[key] = rec
            changed = True
        rows.append((rec["raw_sha256"], rec["decoded_fp"]))

    out = df.copy()
    out["raw_sha256"] = [x[0] for x in rows]
    out["decoded_fp"] = [x[1] for x in rows]

    if changed:
        pd.DataFrame([
            {
                "image_name": k,
                "raw_sha256": v["raw_sha256"],
                "decoded_fp": v["decoded_fp"],
            }
            for k, v in _fp_cache.items()
        ]).to_csv(FP_CACHE, index=False)

    return out

def dedupe_content(df):
    before = len(df)
    x = (
        df.sort_values(["Label", "image_name"])
          .drop_duplicates("decoded_fp", keep="first")
          .reset_index(drop=True)
    )
    print(f"Decoded-content duplicates removed: {before-len(x)}")
    return x

def build_primary_cohort():
    pos = meta[meta.Label == 1].copy()
    neg = meta[meta.Label == 0].copy()

    n_neg_cand = min(len(neg), PRIMARY_PER_CLASS * NEGATIVE_CANDIDATE_MULTIPLIER)
    candidate = pd.concat([
        pos,
        neg.sample(n=n_neg_cand, random_state=SEED)
    ], ignore_index=True)

    print(f"Primary fingerprint candidate pool: {len(candidate):,}")
    candidate = dedupe_content(fingerprint_rows(candidate))

    pos_u = candidate[candidate.Label == 1].copy()
    neg_u = candidate[candidate.Label == 0].copy()

    k = min(PRIMARY_PER_CLASS, len(pos_u), len(neg_u))
    if k < 500:
        raise RuntimeError(f"Too few unique images per class after duplicate audit: {k}")

    psel = pos_u.sample(n=k, random_state=SEED + 1) if len(pos_u) > k else pos_u
    nsel = neg_u.sample(n=k, random_state=SEED + 2)

    cohort = pd.concat([psel, nsel], ignore_index=True)
    cohort = cohort.sample(frac=1, random_state=SEED + 3).reset_index(drop=True)
    cohort["row_id"] = np.arange(len(cohort), dtype=int)
    return cohort

cohort = build_primary_cohort()

print(f"\nPRIMARY COHORT: {len(cohort)} images | {cohort.Patient_ID.nunique()} patients")
print(cohort.Label.value_counts().sort_index())
print("Decoded duplicates remaining:", int(cohort.duplicated("decoded_fp").sum()))
print("Raw-byte duplicates remaining:", int(cohort.duplicated("raw_sha256").sum()))
cohort.to_csv(OUT / "primary_cohort_manifest.csv", index=False)

pat = cohort.groupby("Patient_ID").agg(
    n_images=("image_name", "size"),
    n_labels=("Label", "nunique"),
).reset_index()

structure = pd.DataFrame({
    "images": [len(cohort)],
    "patients": [cohort.Patient_ID.nunique()],
    "multi_image_patients": [int((pat.n_images > 1).sum())],
    "patients_with_both_binary_labels": [int((pat.n_labels > 1).sum())],
    "max_images_per_patient": [int(pat.n_images.max())],
    "median_images_per_patient": [float(pat.n_images.median())],
    "lesion_id_available": [bool(LESION_ID_AVAILABLE)],
    "lesion_id_coverage_pct": [100 * float(cohort.Lesion_ID.notna().mean())],
})
structure.to_csv(OUT / "primary_patient_structure.csv", index=False)
print(structure.to_string(index=False))

Primary fingerprint candidate pool: 2,336
Decoded-content duplicates removed: 3

PRIMARY COHORT: 1162 images | 796 patients
Label
0    581
1    581
Name: count, dtype: int64
Decoded duplicates remaining: 0
Raw-byte duplicates remaining: 0
 images  patients  multi_image_patients  patients_with_both_binary_labels  max_images_per_patient  median_images_per_patient  lesion_id_available  lesion_id_coverage_pct
   1162       796                   241                                88                       8                        1.0                 True                   100.0


In [5]:
# =========================
# 5. Final controlled split builder:
#    exact target+diagnosis+site+sex matching, nearest-age replacement
# =========================
def safe_val_split(safe_df, fold_seed):
    n_groups = safe_df.Patient_ID.nunique()
    n_splits = min(VAL_FOLDS, n_groups)
    if n_splits < 2:
        raise RuntimeError("Not enough safe patient groups for validation.")

    for off in range(50):
        sg = StratifiedGroupKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=fold_seed + off,
        )
        tr_i, va_i = next(
            sg.split(safe_df, safe_df.Label, groups=safe_df.Patient_ID)
        )
        tr = safe_df.iloc[tr_i].copy()
        va = safe_df.iloc[va_i].copy()
        if tr.Label.nunique() == 2 and va.Label.nunique() == 2:
            return tr.reset_index(drop=True), va.reset_index(drop=True)

    raise RuntimeError("Could not construct a two-class patient-disjoint validation split.")

def choose_age_matched_removals(base, candidates):
    # All rows already share target + diagnosis + site + sex exactly.
    available = base.copy()
    removed = []

    for _, c in candidates.sort_values(["AgeApprox", "row_id"], na_position="last").iterrows():
        if len(available) == 0:
            break

        if np.isfinite(c.AgeApprox) and available.AgeApprox.notna().any():
            dist = (available.AgeApprox - c.AgeApprox).abs()
            idx = dist.sort_values(kind="mergesort").index[0]
        else:
            idx = available.sort_values("row_id").index[0]

        removed.append(int(available.loc[idx, "row_id"]))
        available = available.drop(index=idx)

    return removed

def age_smd(a, b):
    aa = a.AgeApprox.dropna()
    bb = b.AgeApprox.dropna()
    if len(aa) < 2 or len(bb) < 2:
        return np.nan
    pooled = np.sqrt((aa.var(ddof=1) + bb.var(ddof=1)) / 2)
    return float((bb.mean() - aa.mean()) / pooled) if pooled > 0 else 0.0

def lesion_exposure_columns(test, leak):
    test = test.copy()
    patient_counts = leak.Patient_ID.astype(str).value_counts()
    same_label = leak.groupby(["Patient_ID", "Label"]).size()

    test["n_train_siblings"] = (
        test.Patient_ID.astype(str).map(patient_counts).fillna(0).astype(int)
    )
    test["n_same_label_siblings"] = [
        int(same_label.get((str(p), int(y)), 0))
        for p, y in zip(test.Patient_ID, test.Label)
    ]
    test["n_opposite_label_siblings"] = [
        int(same_label.get((str(p), 1-int(y)), 0))
        for p, y in zip(test.Patient_ID, test.Label)
    ]

    if LESION_ID_AVAILABLE and leak.Lesion_ID.notna().any():
        same_lesion = leak.dropna(subset=["Lesion_ID"]).groupby("Lesion_ID").size()

        same_vals = []
        diff_vals = []
        types = []

        for _, r in test.iterrows():
            total = int(r.n_train_siblings)
            lid = r.Lesion_ID

            if total == 0:
                same_vals.append(0)
                diff_vals.append(0)
                types.append("no_patient_sibling")
                continue

            if pd.isna(lid):
                same_vals.append(np.nan)
                diff_vals.append(np.nan)
                types.append("patient_sibling_lesion_unknown")
                continue

            same = int(same_lesion.get(str(lid), 0))
            patient_rows = leak[leak.Patient_ID.astype(str) == str(r.Patient_ID)]
            known_diff = patient_rows[
                patient_rows.Lesion_ID.notna()
                & (patient_rows.Lesion_ID.astype(str) != str(lid))
            ]
            diff = int(len(known_diff))

            same_vals.append(same)
            diff_vals.append(diff)

            if same > 0 and diff > 0:
                types.append("mixed_same_and_different_lesion")
            elif same > 0:
                types.append("same_lesion_only")
            elif diff > 0:
                types.append("different_lesion_only")
            else:
                types.append("patient_sibling_lesion_unknown")

        test["n_same_lesion_siblings"] = same_vals
        test["n_different_lesion_siblings"] = diff_vals
        test["LesionExposureType"] = types
    else:
        test["n_same_lesion_siblings"] = np.nan
        test["n_different_lesion_siblings"] = np.nan
        test["LesionExposureType"] = np.where(
            test.n_train_siblings > 0,
            "patient_sibling_lesion_id_unavailable",
            "no_patient_sibling",
        )

    return test

def build_controlled_fold(df, train_idx, test_idx, fold):
    test = df.iloc[test_idx].copy().reset_index(drop=True)
    raw_train = df.iloc[train_idx].copy().reset_index(drop=True)

    test_rows = set(test.row_id.astype(int))
    test_raw = set(test.raw_sha256.astype(str))
    test_dec = set(test.decoded_fp.astype(str))
    test_patients = set(test.Patient_ID.astype(str))

    # Exact/decoded test content is forbidden from both arms.
    raw_train = raw_train[
        (~raw_train.row_id.astype(int).isin(test_rows))
        & (~raw_train.raw_sha256.astype(str).isin(test_raw))
        & (~raw_train.decoded_fp.astype(str).isin(test_dec))
    ].copy()

    safe = raw_train[
        ~raw_train.Patient_ID.astype(str).isin(test_patients)
    ].copy()

    candidates = raw_train[
        raw_train.Patient_ID.astype(str).isin(test_patients)
    ].copy()

    rig, val = safe_val_split(safe, SEED + 1000*fold)

    remove_ids = []
    leak_parts = []
    used = 0
    unmatched = 0

    for stratum, cg in candidates.groupby("MatchStratum", sort=True):
        bg = rig[rig.MatchStratum == stratum].copy()
        if len(bg) == 0:
            unmatched += len(cg)
            continue

        n = min(
            int(round(len(cg) * LEAK_FRACTION)),
            len(cg),
            len(bg),
        )
        if n <= 0:
            unmatched += len(cg)
            continue

        # Deterministic candidate selection; within the exact stratum removals are age matched.
        chosen = cg.sort_values(["AgeApprox", "row_id"], na_position="last").head(n).copy()
        removed = choose_age_matched_removals(bg, chosen)

        if len(removed) != len(chosen):
            raise RuntimeError("Age-matched removal count mismatch.")

        leak_parts.append(chosen)
        remove_ids.extend(removed)
        used += len(chosen)
        unmatched += len(cg) - len(chosen)

    leak = rig[
        ~rig.row_id.astype(int).isin(remove_ids)
    ].copy()

    if leak_parts:
        leak = pd.concat([leak] + leak_parts, ignore_index=True)

    leak = leak.sample(
        frac=1, random_state=SEED + 10000*fold + 99
    ).reset_index(drop=True)

    def ids(x): return set(x.row_id.astype(int))
    def pats(x): return set(x.Patient_ID.astype(str))
    def raws(x): return set(x.raw_sha256.astype(str))
    def decs(x): return set(x.decoded_fp.astype(str))

    assert ids(test).isdisjoint(ids(rig))
    assert ids(test).isdisjoint(ids(leak))
    assert raws(test).isdisjoint(raws(rig))
    assert raws(test).isdisjoint(raws(leak))
    assert decs(test).isdisjoint(decs(rig))
    assert decs(test).isdisjoint(decs(leak))
    assert pats(test).isdisjoint(pats(rig))
    assert pats(test).isdisjoint(pats(val))
    assert pats(val).isdisjoint(pats(rig))
    assert pats(val).isdisjoint(pats(leak))
    assert len(rig) == len(leak)

    # Exact confound-control assertion.
    rig_strata = rig.MatchStratum.value_counts().sort_index()
    leak_strata = leak.MatchStratum.value_counts().sort_index()
    assert rig_strata.equals(leak_strata)

    # Binary counts must also be identical.
    assert (
        rig.Label.value_counts().sort_index().to_dict()
        == leak.Label.value_counts().sort_index().to_dict()
    )

    test = lesion_exposure_columns(test, leak)

    audit = {
        "fold": fold,
        "test_n": len(test),
        "test_patients": test.Patient_ID.nunique(),
        "rig_train_n": len(rig),
        "leak_train_n": len(leak),
        "val_n": len(val),
        "eligible_sibling_candidates": len(candidates),
        "used_sibling_candidates": used,
        "unmatched_sibling_candidates": unmatched,
        "candidate_utilization_pct": 100*used/max(len(candidates), 1),
        "rig_train_test_patient_overlap": len(pats(rig)&pats(test)),
        "leak_train_test_patient_overlap": len(pats(leak)&pats(test)),
        "rig_train_test_rawhash_overlap": len(raws(rig)&raws(test)),
        "leak_train_test_rawhash_overlap": len(raws(leak)&raws(test)),
        "rig_train_test_decoded_overlap": len(decs(rig)&decs(test)),
        "leak_train_test_decoded_overlap": len(decs(leak)&decs(test)),
        "exact_match_strata_preserved": bool(rig_strata.equals(leak_strata)),
        "test_exposure_pct": float((test.n_train_siblings > 0).mean()),
        "same_label_exposure_pct": float((test.n_same_label_siblings > 0).mean()),
        "opposite_label_exposure_pct": float((test.n_opposite_label_siblings > 0).mean()),
        "age_smd_leak_minus_rig": age_smd(rig, leak),
        "lesion_id_available": bool(LESION_ID_AVAILABLE),
    }

    if LESION_ID_AVAILABLE:
        audit["same_lesion_exposure_pct"] = float(
            pd.to_numeric(test.n_same_lesion_siblings, errors="coerce").fillna(0).gt(0).mean()
        )
        audit["different_lesion_exposure_pct"] = float(
            pd.to_numeric(test.n_different_lesion_siblings, errors="coerce").fillna(0).gt(0).mean()
        )

    return rig, leak, val, test, audit

# Synthetic self-test before touching GPU training.
def split_self_test():
    rows = []
    rid = 0
    for p in range(80):
        for j in range(3):
            y = int((p + j) % 2)
            site = "torso" if p % 2 == 0 else "lower extremity"
            sex = "male" if p % 3 else "female"
            diag = "melanoma" if y else "nevus"
            rows.append({
                "row_id": rid,
                "Patient_ID": str(p),
                "Label": y,
                "image_name": f"I{rid}",
                "filepath": f"/tmp/I{rid}.jpg",
                "raw_sha256": f"r{rid}",
                "decoded_fp": f"d{rid}",
                "SiteNorm": site,
                "SexNorm": sex,
                "DiagnosisNorm": diag,
                "AgeApprox": float(30 + (p % 8)*5),
                "MatchStratum": f"Y={y}||DIAG={diag}||SITE={site}||SEX={sex}",
                "Lesion_ID": f"L{p}_{j//2}",
            })
            rid += 1

    d = pd.DataFrame(rows)
    sk = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    tr, te = next(sk.split(d, d.Label))
    a, b, v, t, au = build_controlled_fold(d, tr, te, 1)

    assert au["rig_train_test_patient_overlap"] == 0
    assert au["leak_train_test_rawhash_overlap"] == 0
    assert au["leak_train_test_decoded_overlap"] == 0
    assert au["exact_match_strata_preserved"]
    assert len(a) == len(b)
    print("Synthetic final split self-test: PASSED")

split_self_test()

Synthetic final split self-test: PASSED


In [6]:
# =========================
# 6. Build and freeze five final paired primary folds
# =========================
# Outer image-stratified folds define fixed test images for this controlled exposure experiment.
outer = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)

FOLD_DATA = {}
audit_rows = []
seen_test = []

for fold, (tr_idx, te_idx) in enumerate(outer.split(cohort, cohort.Label), 1):
    rig, leak, val, test, audit = build_controlled_fold(
        cohort, tr_idx, te_idx, fold
    )

    FOLD_DATA[fold] = {
        "rig": rig,
        "leak": leak,
        "val": val,
        "test": test,
    }
    audit_rows.append(audit)
    seen_test.extend(test.row_id.astype(int).tolist())

    for role, frame in [
        ("rig_train", rig),
        ("leak_train", leak),
        ("val", val),
        ("test", test),
    ]:
        x = frame.copy()
        x["fold"] = fold
        x["role"] = role
        x.to_csv(SPLIT_DIR / f"primary_fold{fold}_{role}.csv", index=False)

assert sorted(seen_test) == sorted(cohort.row_id.astype(int).tolist())

audit_df = pd.DataFrame(audit_rows)
audit_df.to_csv(OUT / "primary_final_leakage_audit_by_fold.csv", index=False)

print(audit_df.to_string(index=False))

assert audit_df.rig_train_test_patient_overlap.eq(0).all()
assert audit_df.rig_train_test_rawhash_overlap.eq(0).all()
assert audit_df.leak_train_test_rawhash_overlap.eq(0).all()
assert audit_df.rig_train_test_decoded_overlap.eq(0).all()
assert audit_df.leak_train_test_decoded_overlap.eq(0).all()
assert audit_df.exact_match_strata_preserved.all()

if audit_df.leak_train_test_patient_overlap.eq(0).any():
    raise RuntimeError("At least one fold has no deliberate same-patient exposure.")

print("\nFINAL PRIMARY SPLIT AUDIT: PASSED")
print(f"Mean test-image sibling exposure: {100*audit_df.test_exposure_pct.mean():.1f}%")
print(f"Mean candidate utilization: {audit_df.candidate_utilization_pct.mean():.1f}%")
print(f"Mean age SMD (leak-rig): {audit_df.age_smd_leak_minus_rig.mean():.4f}")

if 100*audit_df.test_exposure_pct.mean() < 20:
    print("WARNING: controlled exposure is below 20%; interpret effect estimates as low-exposure.")

 fold  test_n  test_patients  rig_train_n  leak_train_n  val_n  eligible_sibling_candidates  used_sibling_candidates  unmatched_sibling_candidates  candidate_utilization_pct  rig_train_test_patient_overlap  leak_train_test_patient_overlap  rig_train_test_rawhash_overlap  leak_train_test_rawhash_overlap  rig_train_test_decoded_overlap  leak_train_test_decoded_overlap  exact_match_strata_preserved  test_exposure_pct  same_label_exposure_pct  opposite_label_exposure_pct  age_smd_leak_minus_rig  lesion_id_available  same_lesion_exposure_pct  different_lesion_exposure_pct
    1     233            210          617           617    153                          159                      157                             2                  98.742138                               0                               94                               0                                0                               0                                0                          True           0.459227         

In [7]:
# =========================
# 7. Prediction-independent corrected XAI subset
# =========================
def sample_xai_cases(test, fold, max_n=XAI_MAX_PER_FOLD):
    x = test.copy()

    # Stratify by target, site, and exposure status; this is known before predictions.
    x["ExposureFlag"] = (x.n_train_siblings > 0).astype(int)
    x["XAI_Stratum"] = (
        x.Label.astype(str)
        + "||SITE=" + x.SiteNorm.astype(str)
        + "||EXP=" + x.ExposureFlag.astype(str)
    )

    groups = list(x.groupby("XAI_Stratum", sort=True))
    if not groups:
        return x.iloc[0:0].copy()

    quota = max(1, max_n // len(groups))
    parts = []
    used = set()

    for gi, (_, g) in enumerate(groups):
        n = min(quota, len(g))
        s = g.sample(n=n, random_state=SEED + 1000*fold + gi)
        parts.append(s)
        used.update(s.row_id.astype(int).tolist())

    out = pd.concat(parts, ignore_index=True) if parts else x.iloc[0:0].copy()

    remaining = max_n - len(out)
    if remaining > 0:
        rem = x[~x.row_id.astype(int).isin(used)]
        if len(rem):
            extra = rem.sample(
                n=min(remaining, len(rem)),
                random_state=SEED + 1000*fold + 999,
            )
            out = pd.concat([out, extra], ignore_index=True)

    return out.reset_index(drop=True)

XAI_TEST_BY_FOLD = {}
all_xai_selection = []

for fold in range(1, FOLDS+1):
    x = sample_xai_cases(FOLD_DATA[fold]["test"], fold)
    x["fold"] = fold
    XAI_TEST_BY_FOLD[fold] = x
    all_xai_selection.append(x)

xai_selected = pd.concat(all_xai_selection, ignore_index=True)
xai_selected.to_csv(
    XAI_DIR / "preselected_quantitative_xai_cases.csv",
    index=False,
)

# A tiny fixed visual subset, selected before model predictions.
vis_parts = []
vis_groups = list(xai_selected.groupby(["Label", "ExposureFlag"], sort=True))
quota = max(1, XAI_VISUAL_TOTAL // max(len(vis_groups), 1))

for i, (_, g) in enumerate(vis_groups):
    n = min(quota, len(g))
    vis_parts.append(g.sample(n=n, random_state=SEED + 8000 + i))

visual_cases = pd.concat(vis_parts, ignore_index=True) if vis_parts else xai_selected.head(0)

if len(visual_cases) < XAI_VISUAL_TOTAL:
    used = set(visual_cases.row_id.astype(int).tolist())
    rem = xai_selected[~xai_selected.row_id.astype(int).isin(used)]
    if len(rem):
        extra = rem.sample(
            n=min(XAI_VISUAL_TOTAL-len(visual_cases), len(rem)),
            random_state=SEED + 8999,
        )
        visual_cases = pd.concat([visual_cases, extra], ignore_index=True)

VISUAL_IDS = set(visual_cases.row_id.astype(int).tolist())
visual_cases.to_csv(XAI_DIR / "preselected_visual_cases.csv", index=False)

print("Quantitative XAI cases:", len(xai_selected))
print("Visual XAI cases:", len(visual_cases))

Quantitative XAI cases: 500
Visual XAI cases: 16


In [8]:
# =========================
# 8. Dermoscopy data pipeline
# =========================
IM_MEAN = (0.485, 0.456, 0.406)
IM_STD = (0.229, 0.224, 0.225)

TRAIN_TF = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.RandomRotation(90),
    T.ColorJitter(brightness=0.15, contrast=0.15),
    T.ToTensor(),
    T.Normalize(IM_MEAN, IM_STD),
])

EVAL_TF = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(IM_MEAN, IM_STD),
])

class ISICDataset(Dataset):
    def __init__(self, df, train=False):
        self.df = df.reset_index(drop=True)
        self.tf = TRAIN_TF if train else EVAL_TF

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        with Image.open(r.filepath) as im:
            im = im.convert("RGB")
            x = self.tf(im)
        return x, int(r.Label), int(r.row_id), str(r.Patient_ID)

def make_loader(df, train=False, seed=SEED):
    g = torch.Generator()
    g.manual_seed(seed)

    return DataLoader(
        ISICDataset(df, train=train),
        batch_size=BATCH_SIZE,
        shuffle=train,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
        persistent_workers=False,
        generator=g if train else None,
    )

print("Data pipeline ready.")

Data pipeline ready.


In [9]:
# =========================
# 9. Models + deterministic training + validation-locked threshold
# =========================
def make_resnet50():
    w = tvm.ResNet50_Weights.DEFAULT if PRETRAINED else None
    m = tvm.resnet50(weights=w)
    m.fc = nn.Linear(m.fc.in_features, 2)
    return m

def make_convnext_tiny():
    w = tvm.ConvNeXt_Tiny_Weights.DEFAULT if PRETRAINED else None
    m = tvm.convnext_tiny(weights=w)
    m.classifier[2] = nn.Linear(m.classifier[2].in_features, 2)
    return m

MODEL_FACTORIES = {
    "ResNet50": make_resnet50,
    "ConvNeXtTiny": make_convnext_tiny,
}

for name in RUN_ARCHITECTURES:
    try:
        m = MODEL_FACTORIES[name]()
        del m
    except Exception as e:
        raise RuntimeError(
            f"Could not initialize ImageNet weights for {name}. "
            f"Enable Internet for the first run or provide cached torchvision weights. "
            f"Original error: {repr(e)}"
        )
print("Pretrained-weight preflight: PASSED")

def autocast_ctx():
    return torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=USE_AMP,
    )

@torch.no_grad()
def predict_prob(model, loader):
    rows = []
    model.eval()

    for x, y, row_id, patient in loader:
        x = x.to(DEVICE, non_blocking=True)
        with autocast_ctx():
            logits = model(x)

        p = torch.softmax(logits.float(), dim=1)[:, 1].cpu().numpy()

        for yy, pp, rid, pat in zip(y.numpy(), p, row_id.numpy(), patient):
            rows.append({
                "row_id": int(rid),
                "Patient_ID": str(pat),
                "Label": int(yy),
                "prob": float(pp),
            })

    return pd.DataFrame(rows)

def choose_youden_threshold(y, p):
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)

    if len(np.unique(y)) < 2:
        return 0.5

    fpr, tpr, thr = roc_curve(y, p)
    finite = np.isfinite(thr)
    fpr, tpr, thr = fpr[finite], tpr[finite], thr[finite]

    if len(thr) == 0:
        return 0.5

    j = tpr - fpr
    best = np.flatnonzero(j == np.nanmax(j))
    idx = best[np.argmin(np.abs(thr[best] - 0.5))]
    return float(np.clip(thr[idx], 0.0, 1.0))

def class_weights(df):
    c = df.Label.value_counts()
    n = len(df)
    return torch.tensor([
        n / (2*max(int(c.get(0, 1)), 1)),
        n / (2*max(int(c.get(1, 1)), 1)),
    ], dtype=torch.float32, device=DEVICE)

def train_one(
    model_name, train_df, val_df, fold, arm,
    class_weight=None, experiment="primary"
):
    model_offset = 0 if model_name == "ResNet50" else 10
    exp_offset = 0 if experiment == "primary" else 10000
    run_seed = SEED + 100*fold + model_offset + exp_offset

    seed_everything(run_seed)

    model = MODEL_FACTORIES[model_name]().to(DEVICE)
    trL = make_loader(train_df, train=True, seed=run_seed)
    vaL = make_loader(val_df, train=False, seed=run_seed)

    opt = torch.optim.Adam(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=EPOCHS
    )
    loss_fn = nn.CrossEntropyLoss(weight=class_weight)

    if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
        try:
            scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
        except TypeError:
            scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
    else:
        scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    best_auc = -np.inf
    best_state = None
    best_epoch = None
    wait = 0
    hist = []

    print(
        f"{experiment} | {model_name} | fold {fold} | {arm} | "
        f"train={len(train_df)} val={len(val_df)}"
    )

    for ep in range(1, EPOCHS+1):
        t0 = time.time()
        model.train()
        total_loss = 0.0
        n = 0

        for x, y, _, _ in trL:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)

            with autocast_ctx():
                logits = model(x)
                loss = loss_fn(logits, y)

            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

            total_loss += float(loss.detach()) * len(y)
            n += len(y)

        sch.step()

        vp = predict_prob(model, vaL)
        vauc = (
            roc_auc_score(vp.Label, vp.prob)
            if vp.Label.nunique() == 2 else np.nan
        )

        hist.append({
            "epoch": ep,
            "train_loss": total_loss/max(n, 1),
            "val_auc": vauc,
        })

        print(
            f"  epoch {ep:02d}/{EPOCHS} "
            f"loss={total_loss/max(n,1):.4f} "
            f"valAUC={vauc:.4f} "
            f"time={time.time()-t0:.0f}s"
        )

        if np.isfinite(vauc) and vauc > best_auc + 1e-6:
            best_auc = vauc
            best_epoch = ep
            wait = 0
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
        else:
            wait += 1
            if wait >= PATIENCE:
                print("  early stop")
                break

    if best_state is None:
        raise RuntimeError(
            f"No valid validation checkpoint: {experiment} {model_name} fold {fold} {arm}"
        )

    model.load_state_dict(best_state)
    model.eval()

    vp = predict_prob(
        model,
        make_loader(val_df, train=False, seed=run_seed)
    )
    locked_threshold = choose_youden_threshold(
        vp.Label, vp.prob
    )

    return model, pd.DataFrame(hist), int(best_epoch), float(locked_threshold), vp

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 201MB/s]


Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 198MB/s] 


Pretrained-weight preflight: PASSED


In [10]:
# =========================
# 10. Dependency-free ResNet-50 Grad-CAM
# =========================
def gradcam_native(model, x, targets):
    activations = {}
    gradients = {}
    layer = model.layer4

    def fwd_hook(module, inp, out):
        activations["value"] = out

    def bwd_hook(module, grad_input, grad_output):
        gradients["value"] = grad_output[0]

    h1 = layer.register_forward_hook(fwd_hook)
    h2 = layer.register_full_backward_hook(bwd_hook)

    model.zero_grad(set_to_none=True)
    logits = model(x)
    score = logits.gather(1, targets.view(-1, 1)).sum()
    score.backward()

    A = activations["value"]
    G = gradients["value"]
    w = G.mean(dim=(2,3), keepdim=True)
    cam = torch.relu((w*A).sum(dim=1))

    flat = cam.flatten(1)
    mn = flat.min(dim=1).values[:, None, None]
    mx = flat.max(dim=1).values[:, None, None]
    cam = (cam-mn)/(mx-mn+1e-8)

    h1.remove()
    h2.remove()
    return cam.detach()

def cam_centroid(cam):
    H, W = cam.shape
    mass = float(cam.sum()) + 1e-12
    yy, xx = np.mgrid[0:H, 0:W]
    cx = float((cam*xx).sum()/mass) / max(W-1, 1)
    cy = float((cam*yy).sum()/mass) / max(H-1, 1)
    return cx, cy

def cam_pair_metrics(a, b):
    af = a.reshape(-1).astype(float)
    bf = b.reshape(-1).astype(float)

    denom = np.linalg.norm(af)*np.linalg.norm(bf)
    cosine = float(np.dot(af, bf)/denom) if denom > 0 else np.nan

    if np.std(af) > 0 and np.std(bf) > 0:
        corr = float(np.corrcoef(af, bf)[0,1])
    else:
        corr = np.nan

    ax, ay = cam_centroid(a)
    bx, by = cam_centroid(b)
    shift = float(
        np.sqrt((ax-bx)**2 + (ay-by)**2) / np.sqrt(2)
    )

    return cosine, corr, shift

def denormalize(x):
    mean = torch.tensor(IM_MEAN, device=x.device).view(3,1,1)
    std = torch.tensor(IM_STD, device=x.device).view(3,1,1)
    return torch.clamp(x*std + mean, 0, 1)

def save_overlay(x, cam, path, title):
    base = denormalize(x.detach()).cpu().permute(1,2,0).numpy()
    c = torch.tensor(cam)[None, None]
    c = torch.nn.functional.interpolate(
        c,
        size=(IMG_SIZE, IMG_SIZE),
        mode="bilinear",
        align_corners=False,
    )[0,0].numpy()

    fig, ax = plt.subplots(figsize=(4,4))
    ax.imshow(base)
    ax.imshow(c, cmap="jet", alpha=0.42, vmin=0, vmax=1)
    ax.axis("off")
    ax.set_title(title, fontsize=8)
    fig.tight_layout()
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)

def extract_xai(model, xai_df, fold, arm):
    loader = make_loader(
        xai_df,
        train=False,
        seed=SEED + fold,
    )
    cams = {}
    meta_out = {}
    model.eval()

    for x, y, row_id, patient in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        # Full precision for attribution gradients.
        c = gradcam_native(model, x, y).cpu().numpy()

        for j, rid in enumerate(row_id.numpy()):
            rid = int(rid)
            cams[rid] = c[j].astype(np.float32)

            rr = xai_df[
                xai_df.row_id.astype(int) == rid
            ].iloc[0]

            meta_out[rid] = {
                "Patient_ID": str(rr.Patient_ID),
                "Label": int(rr.Label),
                "SiteNorm": str(rr.SiteNorm),
                "LesionExposureType": str(rr.LesionExposureType),
                "n_train_siblings": int(rr.n_train_siblings),
            }

            if rid in VISUAL_IDS:
                save_overlay(
                    x[j],
                    c[j],
                    XAI_DIR / f"row{rid}_fold{fold}_{arm}.png",
                    (
                        f"fold {fold} | {arm} | y={int(rr.Label)} | "
                        f"site={rr.SiteNorm}"
                    ),
                )

    return cams, meta_out

In [11]:
# =========================
# 11. Final deterministic paired 5-fold primary rerun + corrected XAI
# =========================
all_oof = []
training_records = []
xai_pair_rows = []

for model_name in RUN_ARCHITECTURES:
    print("\n" + "="*90)
    print("PRIMARY ARCHITECTURE:", model_name)
    print("="*90)

    for fold in range(1, FOLDS+1):
        D = FOLD_DATA[fold]
        test = D["test"].copy()
        arm_predictions = {}
        rig_cams = None
        rig_meta = None

        for arm, key in [
            ("rigorous", "rig"),
            ("controlled_leak", "leak"),
        ]:
            model, hist, best_epoch, threshold, val_pred = train_one(
                model_name,
                D[key],
                D["val"],
                fold,
                arm,
                class_weight=None,
                experiment="primary",
            )

            pp = predict_prob(
                model,
                make_loader(test, train=False, seed=SEED+fold),
            )
            pp["pred05"] = (pp.prob >= 0.5).astype(int)
            pp["pred_locked"] = (pp.prob >= threshold).astype(int)
            pp["locked_threshold"] = threshold

            pp.to_csv(
                PRED_DIR / f"primary_{model_name}_fold{fold}_{arm}.csv",
                index=False,
            )
            hist.to_csv(
                PRED_DIR / f"primary_{model_name}_fold{fold}_{arm}_history.csv",
                index=False,
            )

            val_pred.assign(
                pred_locked=(val_pred.prob >= threshold).astype(int),
                locked_threshold=threshold,
            ).to_csv(
                PRED_DIR / f"primary_{model_name}_fold{fold}_{arm}_validation_predictions.csv",
                index=False,
            )

            arm_predictions[arm] = pp
            training_records.append({
                "experiment": "primary",
                "architecture": model_name,
                "fold": fold,
                "arm": arm,
                "best_epoch": best_epoch,
                "locked_threshold": threshold,
                "train_n": len(D[key]),
                "val_n": len(D["val"]),
                "test_n": len(test),
            })

            if model_name == XAI_ARCHITECTURE:
                cams, mm = extract_xai(
                    model,
                    XAI_TEST_BY_FOLD[fold],
                    fold,
                    arm,
                )

                if arm == "rigorous":
                    rig_cams = cams
                    rig_meta = mm
                else:
                    common = sorted(set(rig_cams) & set(cams))
                    for rid in common:
                        cos, corr, shift = cam_pair_metrics(
                            rig_cams[rid],
                            cams[rid],
                        )
                        m = mm[rid]
                        xai_pair_rows.append({
                            "row_id": rid,
                            "fold": fold,
                            "Patient_ID": m["Patient_ID"],
                            "Label": m["Label"],
                            "SiteNorm": m["SiteNorm"],
                            "LesionExposureType": m["LesionExposureType"],
                            "n_train_siblings": m["n_train_siblings"],
                            "exposed": int(m["n_train_siblings"] > 0),
                            "cam_cosine_similarity": cos,
                            "cam_pearson_r": corr,
                            "cam_centroid_shift": shift,
                        })

            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        r = arm_predictions["rigorous"].rename(columns={
            "prob": "prob_rig",
            "pred05": "pred05_rig",
            "pred_locked": "pred_locked_rig",
            "locked_threshold": "threshold_rig",
        })

        l = arm_predictions["controlled_leak"][[
            "row_id", "prob", "pred05", "pred_locked", "locked_threshold"
        ]].rename(columns={
            "prob": "prob_leak",
            "pred05": "pred05_leak",
            "pred_locked": "pred_locked_leak",
            "locked_threshold": "threshold_leak",
        })

        keep = [
            "row_id", "Patient_ID", "Label", "image_name",
            "DiagnosisNorm", "SiteNorm", "SexNorm", "AgeApprox",
            "Lesion_ID", "n_train_siblings", "n_same_label_siblings",
            "n_opposite_label_siblings", "n_same_lesion_siblings",
            "n_different_lesion_siblings", "LesionExposureType",
        ]

        merged = test[keep].merge(
            r[[
                "row_id", "prob_rig", "pred05_rig",
                "pred_locked_rig", "threshold_rig"
            ]],
            on="row_id",
            how="inner",
        ).merge(
            l,
            on="row_id",
            how="inner",
        )

        assert len(merged) == len(test)
        merged["fold"] = fold
        merged["architecture"] = model_name
        all_oof.append(merged)

oof = pd.concat(all_oof, ignore_index=True)
oof.to_csv(
    OUT / "primary_oof_paired_predictions.csv",
    index=False,
)

training_log = pd.DataFrame(training_records)
training_log.to_csv(
    OUT / "training_summary.csv",
    index=False,
)

xai_pairs = pd.DataFrame(xai_pair_rows)
xai_pairs.to_csv(
    XAI_DIR / "gradcam_paired_map_metrics.csv",
    index=False,
)

print("\nPrimary OOF rows:", len(oof))
print("XAI paired rows:", len(xai_pairs))


PRIMARY ARCHITECTURE: ResNet50
primary | ResNet50 | fold 1 | rigorous | train=617 val=153
  epoch 01/12 loss=0.6379 valAUC=0.7990 time=117s
  epoch 02/12 loss=0.4733 valAUC=0.8619 time=118s
  epoch 03/12 loss=0.3680 valAUC=0.8532 time=113s
  epoch 04/12 loss=0.3353 valAUC=0.8652 time=111s
  epoch 05/12 loss=0.2928 valAUC=0.8502 time=113s
  epoch 06/12 loss=0.2615 valAUC=0.8471 time=113s
  epoch 07/12 loss=0.2278 valAUC=0.8408 time=114s
  early stop
primary | ResNet50 | fold 1 | controlled_leak | train=617 val=153
  epoch 01/12 loss=0.6289 valAUC=0.7724 time=117s
  epoch 02/12 loss=0.4896 valAUC=0.8318 time=129s
  epoch 03/12 loss=0.3786 valAUC=0.8371 time=122s
  epoch 04/12 loss=0.3263 valAUC=0.8471 time=113s
  epoch 05/12 loss=0.2869 valAUC=0.8482 time=119s
  epoch 06/12 loss=0.2461 valAUC=0.8581 time=114s
  epoch 07/12 loss=0.2409 valAUC=0.8722 time=116s
  epoch 08/12 loss=0.1883 valAUC=0.8700 time=115s
  epoch 09/12 loss=0.1858 valAUC=0.8756 time=115s
  epoch 10/12 loss=0.1667 valA

In [12]:
# =========================
# 12. Patient-cluster paired statistics
# =========================
PROB_METRICS = ["AUC", "PR_AUC", "Brier", "LogLoss"]
THRESH_METRICS = [
    "ACC", "Macro_F1", "Balanced_ACC",
    "MCC", "Sensitivity", "Specificity"
]
ALL_METRICS = PROB_METRICS + THRESH_METRICS

def metric_on_df(d, arm, metric, locked=True):
    y = d.Label.to_numpy(dtype=int)
    p = d[f"prob_{arm}"].to_numpy(dtype=float)

    if metric == "AUC":
        return roc_auc_score(y, p) if len(np.unique(y)) == 2 else np.nan
    if metric == "PR_AUC":
        return average_precision_score(y, p) if len(np.unique(y)) == 2 else np.nan
    if metric == "Brier":
        return brier_score_loss(y, p)
    if metric == "LogLoss":
        return log_loss(y, np.c_[1-p, p], labels=[0,1])

    pred_col = (
        f"pred_locked_{arm}" if locked
        else f"pred05_{arm}"
    )
    pred = d[pred_col].to_numpy(dtype=int)

    if metric == "ACC":
        return accuracy_score(y, pred)
    if metric == "Macro_F1":
        return f1_score(y, pred, average="macro", zero_division=0)
    if metric == "Balanced_ACC":
        return balanced_accuracy_score(y, pred)
    if metric == "MCC":
        return matthews_corrcoef(y, pred)
    if metric == "Sensitivity":
        return recall_score(y, pred, pos_label=1, zero_division=0)
    if metric == "Specificity":
        return recall_score(y, pred, pos_label=0, zero_division=0)

    raise KeyError(metric)

def macro_fold_metric(d, arm, metric, locked=True):
    vals = []
    for _, g in d.groupby("fold"):
        v = metric_on_df(g, arm, metric, locked)
        if np.isfinite(v):
            vals.append(v)
    return float(np.mean(vals)) if vals else np.nan

def paired_cluster_bootstrap(
    d, metric, locked=True,
    B=BOOTSTRAPS, seed=SEED
):
    patients = np.array(
        sorted(d.Patient_ID.astype(str).unique())
    )
    byp = {
        p: d[d.Patient_ID.astype(str) == p]
        for p in patients
    }
    rng = np.random.default_rng(seed)
    deltas = []

    for _ in range(B):
        samp = rng.choice(
            patients, size=len(patients), replace=True
        )
        x = pd.concat(
            [byp[p] for p in samp],
            ignore_index=True,
        )

        rv = macro_fold_metric(
            x, "rig", metric, locked
        )
        lv = macro_fold_metric(
            x, "leak", metric, locked
        )

        if np.isfinite(rv) and np.isfinite(lv):
            deltas.append(lv-rv)

    if len(deltas) < max(200, B//2):
        raise RuntimeError(
            f"Too few valid bootstrap replicates for {metric}"
        )

    return (
        float(np.percentile(deltas, 2.5)),
        float(np.percentile(deltas, 97.5)),
        len(deltas),
    )

def cluster_randomization_p(
    d, metric, locked=True,
    B=PERMUTATIONS, seed=SEED+7
):
    obs = (
        macro_fold_metric(d, "leak", metric, locked)
        - macro_fold_metric(d, "rig", metric, locked)
    )

    pats = np.array(
        sorted(d.Patient_ID.astype(str).unique())
    )
    rng = np.random.default_rng(seed)
    extreme = 0
    valid = 0

    for _ in range(B):
        swap = set(
            pats[
                rng.integers(
                    0, 2, size=len(pats)
                ).astype(bool)
            ]
        )

        x = d.copy()
        mask = x.Patient_ID.astype(str).isin(swap).to_numpy()

        for a, b in [
            ("prob_rig", "prob_leak"),
            ("pred_locked_rig", "pred_locked_leak"),
            ("pred05_rig", "pred05_leak"),
        ]:
            av = x[a].to_numpy().copy()
            bv = x[b].to_numpy().copy()
            aa, bb = av.copy(), bv.copy()
            aa[mask] = bv[mask]
            bb[mask] = av[mask]
            x[a], x[b] = aa, bb

        dd = (
            macro_fold_metric(x, "leak", metric, locked)
            - macro_fold_metric(x, "rig", metric, locked)
        )

        if np.isfinite(dd):
            valid += 1
            if abs(dd) >= abs(obs)-1e-15:
                extreme += 1

    return float((extreme+1)/(valid+1))

rows = []

for arch, d in oof.groupby("architecture"):
    for metric in ALL_METRICS:
        rv = macro_fold_metric(d, "rig", metric, True)
        lv = macro_fold_metric(d, "leak", metric, True)

        off = sum(map(ord, arch+metric))
        lo, hi, bn = paired_cluster_bootstrap(
            d, metric, True,
            seed=SEED+off,
        )
        p = cluster_randomization_p(
            d, metric, True,
            seed=SEED+123+off,
        )

        rows.append({
            "Architecture": arch,
            "Metric": metric,
            "ThresholdPolicy": (
                "validation-locked Youden J"
                if metric in THRESH_METRICS
                else "not applicable"
            ),
            "Rigorous": rv,
            "ControlledLeak": lv,
            "Delta_LeakMinusRig": lv-rv,
            "Delta_CI_Lo": lo,
            "Delta_CI_Hi": hi,
            "ClusterRandomization_p": p,
            "BootstrapReplicates": bn,
        })

stats = pd.DataFrame(rows)
stats.to_csv(
    OUT / "primary_cluster_aware_metrics.csv",
    index=False,
)

# Supplementary fixed-0.5 threshold metrics.
fixed_rows = []
for arch, d in oof.groupby("architecture"):
    for metric in THRESH_METRICS:
        rv = macro_fold_metric(d, "rig", metric, False)
        lv = macro_fold_metric(d, "leak", metric, False)
        fixed_rows.append({
            "Architecture": arch,
            "Metric": metric,
            "Rigorous_Threshold0.5": rv,
            "ControlledLeak_Threshold0.5": lv,
            "Delta": lv-rv,
        })

pd.DataFrame(fixed_rows).to_csv(
    OUT / "supplementary_fixed_0p5_threshold_metrics.csv",
    index=False,
)

print("\n=== FINAL PRIMARY CLUSTER-AWARE RESULTS ===")
print(
    stats.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)


=== FINAL PRIMARY CLUSTER-AWARE RESULTS ===
Architecture       Metric            ThresholdPolicy  Rigorous  ControlledLeak  Delta_LeakMinusRig  Delta_CI_Lo  Delta_CI_Hi  ClusterRandomization_p  BootstrapReplicates
ConvNeXtTiny          AUC             not applicable    0.8819          0.8779             -0.0040      -0.0141       0.0063                  0.4973                 2000
ConvNeXtTiny       PR_AUC             not applicable    0.8531          0.8489             -0.0042      -0.0206       0.0098                  0.6722                 2000
ConvNeXtTiny        Brier             not applicable    0.1422          0.1452              0.0030      -0.0061       0.0126                  0.5372                 2000
ConvNeXtTiny      LogLoss             not applicable    0.4607          0.4625              0.0019      -0.0277       0.0320                  0.8951                 2000
ConvNeXtTiny          ACC validation-locked Youden J    0.7952          0.7935             -0.0018      -

In [13]:
# =========================
# 13. Exposure, lesion-level, and corrected XAI analyses
# =========================
def true_class_conf(y, p):
    y = np.asarray(y)
    p = np.asarray(p)
    return np.where(y == 1, p, 1-p)

exposure_rows = []
trend_rows = []

for arch, d in oof.groupby("architecture"):
    z = d.copy()
    z["correct_leak"] = (
        z.pred_locked_leak == z.Label
    ).astype(int)
    z["true_conf_leak"] = true_class_conf(
        z.Label, z.prob_leak
    )

    # Main patient-level exposure groups.
    z["exposed_any"] = (
        z.n_train_siblings > 0
    ).astype(int)

    for ex, g in z.groupby("exposed_any"):
        exposure_rows.append({
            "Architecture": arch,
            "ExposureType": "any_same_patient_sibling",
            "ExposureGroup": int(ex),
            "n_images": len(g),
            "n_patients": g.Patient_ID.nunique(),
            "Accuracy": g.correct_leak.mean(),
            "MeanTrueClassConfidence": g.true_conf_leak.mean(),
            "MeanSiblingCount": g.n_train_siblings.mean(),
        })

    # Lesion-level descriptive groups when available.
    if LESION_ID_AVAILABLE:
        for typ, g in z.groupby("LesionExposureType"):
            exposure_rows.append({
                "Architecture": arch,
                "ExposureType": "lesion_exposure_category",
                "ExposureGroup": str(typ),
                "n_images": len(g),
                "n_patients": g.Patient_ID.nunique(),
                "Accuracy": g.correct_leak.mean(),
                "MeanTrueClassConfidence": g.true_conf_leak.mean(),
                "MeanSiblingCount": g.n_train_siblings.mean(),
            })

    patients = np.array(
        sorted(z.Patient_ID.astype(str).unique())
    )
    byp = {
        p: z[z.Patient_ID.astype(str) == p]
        for p in patients
    }

    exposure_vars = [
        "n_train_siblings",
        "n_same_label_siblings",
        "n_opposite_label_siblings",
    ]

    if LESION_ID_AVAILABLE:
        exposure_vars += [
            "n_same_lesion_siblings",
            "n_different_lesion_siblings",
        ]

    for ev in exposure_vars:
        vv = pd.to_numeric(
            z[ev], errors="coerce"
        )

        if vv.notna().sum() == 0 or vv.nunique(dropna=True) <= 1:
            continue

        rng = np.random.default_rng(
            SEED + 77 + sum(map(ord, ev))
        )
        boot_rho = []

        for _ in range(BOOTSTRAPS):
            samp = rng.choice(
                patients,
                len(patients),
                replace=True,
            )
            x = pd.concat(
                [byp[p] for p in samp],
                ignore_index=True,
            )
            xv = pd.to_numeric(
                x[ev], errors="coerce"
            )
            ok = xv.notna()

            if ok.sum() > 2 and xv[ok].nunique() > 1:
                r, _ = spearmanr(
                    xv[ok],
                    x.loc[ok, "correct_leak"],
                )
                if np.isfinite(r):
                    boot_rho.append(r)

        ok = vv.notna()
        obs, _ = spearmanr(
            vv[ok],
            z.loc[ok, "correct_leak"],
        )

        trend_rows.append({
            "Architecture": arch,
            "ExposureVariable": ev,
            "Spearman_rho": obs,
            "Spearman_CI_Lo": (
                np.percentile(boot_rho, 2.5)
                if boot_rho else np.nan
            ),
            "Spearman_CI_Hi": (
                np.percentile(boot_rho, 97.5)
                if boot_rho else np.nan
            ),
            "BootstrapReplicates": len(boot_rho),
        })

exposure = pd.DataFrame(exposure_rows)
trends = pd.DataFrame(trend_rows)

exposure.to_csv(
    OUT / "primary_exposure_groups.csv",
    index=False,
)
trends.to_csv(
    OUT / "primary_exposure_trends.csv",
    index=False,
)

# ---------- XAI cluster summaries ----------
def cluster_bootstrap_mean(
    df, value_col,
    B=BOOTSTRAPS, seed=SEED
):
    pats = np.array(
        sorted(df.Patient_ID.astype(str).unique())
    )
    byp = {
        p: df[df.Patient_ID.astype(str) == p]
        for p in pats
    }
    rng = np.random.default_rng(seed)
    vals = []

    for _ in range(B):
        samp = rng.choice(
            pats, len(pats), replace=True
        )
        x = pd.concat(
            [byp[p] for p in samp],
            ignore_index=True,
        )
        vals.append(
            float(x[value_col].mean())
        )

    return (
        float(df[value_col].mean()),
        float(np.percentile(vals, 2.5)),
        float(np.percentile(vals, 97.5)),
    )

def cluster_bootstrap_group_difference(
    df, value_col,
    group_col="exposed",
    B=BOOTSTRAPS, seed=SEED+500
):
    pats = np.array(
        sorted(df.Patient_ID.astype(str).unique())
    )
    byp = {
        p: df[df.Patient_ID.astype(str) == p]
        for p in pats
    }
    rng = np.random.default_rng(seed)
    vals = []

    obs = (
        df.loc[df[group_col] == 1, value_col].mean()
        - df.loc[df[group_col] == 0, value_col].mean()
    )

    for _ in range(B):
        samp = rng.choice(
            pats, len(pats), replace=True
        )
        x = pd.concat(
            [byp[p] for p in samp],
            ignore_index=True,
        )
        if (
            (x[group_col] == 1).any()
            and (x[group_col] == 0).any()
        ):
            vals.append(float(
                x.loc[x[group_col] == 1, value_col].mean()
                - x.loc[x[group_col] == 0, value_col].mean()
            ))

    return (
        float(obs),
        float(np.percentile(vals, 2.5)),
        float(np.percentile(vals, 97.5)),
    )

xai_rows = []

for metric in [
    "cam_cosine_similarity",
    "cam_pearson_r",
    "cam_centroid_shift",
]:
    q = xai_pairs.dropna(subset=[metric])

    mean, lo, hi = cluster_bootstrap_mean(
        q,
        metric,
        seed=SEED + 900 + sum(map(ord, metric)),
    )
    diff, dlo, dhi = cluster_bootstrap_group_difference(
        q,
        metric,
        seed=SEED + 1200 + sum(map(ord, metric)),
    )

    xai_rows.append({
        "Metric": metric,
        "n_images": len(q),
        "n_patients": q.Patient_ID.nunique(),
        "OverallMean": mean,
        "Overall_CI_Lo": lo,
        "Overall_CI_Hi": hi,
        "ExposedMinusUnexposed": diff,
        "ExposureDifference_CI_Lo": dlo,
        "ExposureDifference_CI_Hi": dhi,
    })

xai_stats = pd.DataFrame(xai_rows)
xai_stats.to_csv(
    XAI_DIR / "gradcam_cluster_summary.csv",
    index=False,
)

print("\nExposure groups:")
print(exposure.to_string(index=False))
print("\nExposure trends:")
print(trends.to_string(index=False))
print("\nCorrected XAI:")
print(xai_stats.to_string(index=False))


Exposure groups:
Architecture             ExposureType         ExposureGroup  n_images  n_patients  Accuracy  MeanTrueClassConfidence  MeanSiblingCount
ConvNeXtTiny any_same_patient_sibling                     0       633         594  0.789889                 0.748391          0.000000
ConvNeXtTiny any_same_patient_sibling                     1       529         204  0.797732                 0.749770          1.741021
ConvNeXtTiny lesion_exposure_category different_lesion_only       529         204  0.797732                 0.749770          1.741021
ConvNeXtTiny lesion_exposure_category    no_patient_sibling       633         594  0.789889                 0.748391          0.000000
    ResNet50 any_same_patient_sibling                     0       633         594  0.786730                 0.729301          0.000000
    ResNet50 any_same_patient_sibling                     1       529         204  0.803403                 0.741538          1.741021
    ResNet50 lesion_exposure_category

In [14]:
# =========================
# 14. Capped-imbalance sensitivity with the SAME final matching/statistical design
# =========================
imbalance_stats = pd.DataFrame()
imbalance_audit = pd.DataFrame()
imbalance_training = []

if RUN_CAPPED_IMBALANCE_SENSITIVITY:
    pos = meta[meta.Label == 1].copy()
    neg = meta[meta.Label == 0].copy()

    n_neg_cand = min(
        len(neg),
        IMBALANCE_MAX_BENIGN + 1000,
    )

    candidate = pd.concat([
        pos,
        neg.sample(
            n=n_neg_cand,
            random_state=SEED + 500,
        ),
    ], ignore_index=True)

    print(
        "Capped-imbalance fingerprint candidate pool:",
        len(candidate),
    )

    candidate = dedupe_content(
        fingerprint_rows(candidate)
    )

    pos_u = candidate[candidate.Label == 1].copy()
    neg_u = candidate[candidate.Label == 0].copy()
    nneg = min(IMBALANCE_MAX_BENIGN, len(neg_u))

    imb = pd.concat([
        pos_u,
        neg_u.sample(
            n=nneg,
            random_state=SEED + 501,
        ),
    ], ignore_index=True).sample(
        frac=1,
        random_state=SEED + 502,
    ).reset_index(drop=True)

    imb["row_id"] = np.arange(
        10_000_000,
        10_000_000 + len(imb),
        dtype=int,
    )

    imb.to_csv(
        OUT / "capped_imbalance_cohort_manifest.csv",
        index=False,
    )

    print(
        f"CAPPED-IMBALANCE COHORT: {len(imb)} images | "
        f"prevalence={imb.Label.mean():.4%}"
    )
    print(
        "This is a capped sensitivity analysis, NOT native ISIC prevalence."
    )

    sk = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=SEED + 600,
    )
    tr_idx, te_idx = next(
        sk.split(imb, imb.Label)
    )

    irig, ileak, ival, itest, iaudit = build_controlled_fold(
        imb,
        tr_idx,
        te_idx,
        fold=101,
    )

    imbalance_audit = pd.DataFrame([iaudit])
    imbalance_audit.to_csv(
        OUT / "capped_imbalance_final_audit.csv",
        index=False,
    )

    iw = class_weights(irig)
    arm_pred = {}

    for arm, train_df in [
        ("rigorous", irig),
        ("controlled_leak", ileak),
    ]:
        model, hist, best_epoch, threshold, val_pred = train_one(
            IMBALANCE_MODEL,
            train_df,
            ival,
            101,
            arm,
            class_weight=iw,
            experiment="capped_imbalance",
        )

        pp = predict_prob(
            model,
            make_loader(
                itest,
                train=False,
                seed=SEED + 101,
            ),
        )
        pp["pred05"] = (
            pp.prob >= 0.5
        ).astype(int)
        pp["pred_locked"] = (
            pp.prob >= threshold
        ).astype(int)
        pp["locked_threshold"] = threshold

        pp.to_csv(
            PRED_DIR / f"capped_imbalance_{arm}.csv",
            index=False,
        )
        hist.to_csv(
            PRED_DIR / f"capped_imbalance_{arm}_history.csv",
            index=False,
        )

        arm_pred[arm] = pp
        imbalance_training.append({
            "experiment": "capped_imbalance",
            "architecture": IMBALANCE_MODEL,
            "fold": 101,
            "arm": arm,
            "best_epoch": best_epoch,
            "locked_threshold": threshold,
            "train_n": len(train_df),
            "val_n": len(ival),
            "test_n": len(itest),
        })

        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    r = arm_pred["rigorous"].rename(columns={
        "prob": "prob_rig",
        "pred05": "pred05_rig",
        "pred_locked": "pred_locked_rig",
        "locked_threshold": "threshold_rig",
    })

    l = arm_pred["controlled_leak"][[
        "row_id", "prob", "pred05",
        "pred_locked", "locked_threshold"
    ]].rename(columns={
        "prob": "prob_leak",
        "pred05": "pred05_leak",
        "pred_locked": "pred_locked_leak",
        "locked_threshold": "threshold_leak",
    })

    ioof = itest[[
        "row_id", "Patient_ID", "Label",
        "n_train_siblings", "LesionExposureType"
    ]].merge(
        r[[
            "row_id", "prob_rig", "pred05_rig",
            "pred_locked_rig", "threshold_rig"
        ]],
        on="row_id",
        how="inner",
    ).merge(
        l,
        on="row_id",
        how="inner",
    )

    ioof["fold"] = 1
    ioof["architecture"] = IMBALANCE_MODEL
    ioof.to_csv(
        OUT / "capped_imbalance_paired_predictions.csv",
        index=False,
    )

    rows = []

    for metric in ALL_METRICS:
        rv = macro_fold_metric(
            ioof, "rig", metric, True
        )
        lv = macro_fold_metric(
            ioof, "leak", metric, True
        )

        off = sum(map(ord, "imbalance"+metric))
        lo, hi, bn = paired_cluster_bootstrap(
            ioof,
            metric,
            True,
            seed=SEED + 1700 + off,
        )
        p = cluster_randomization_p(
            ioof,
            metric,
            True,
            seed=SEED + 1800 + off,
        )

        rows.append({
            "Architecture": IMBALANCE_MODEL,
            "Metric": metric,
            "ThresholdPolicy": (
                "validation-locked Youden J"
                if metric in THRESH_METRICS
                else "not applicable"
            ),
            "Rigorous": rv,
            "ControlledLeak": lv,
            "Delta_LeakMinusRig": lv-rv,
            "Delta_CI_Lo": lo,
            "Delta_CI_Hi": hi,
            "ClusterRandomization_p": p,
            "ObservedPositivePrevalence": float(imb.Label.mean()),
        })

    imbalance_stats = pd.DataFrame(rows)
    imbalance_stats.to_csv(
        OUT / "capped_imbalance_cluster_aware_metrics.csv",
        index=False,
    )

    print("\n=== CAPPED-IMBALANCE SENSITIVITY ===")
    print(imbalance_audit.to_string(index=False))
    print(
        imbalance_stats.to_string(
            index=False,
            float_format=lambda x: f"{x:.4f}",
        )
    )
else:
    print("Capped-imbalance sensitivity disabled.")

Capped-imbalance fingerprint candidate pool: 4584
Decoded-content duplicates removed: 7
CAPPED-IMBALANCE COHORT: 3581 images | prevalence=16.2245%
This is a capped sensitivity analysis, NOT native ISIC prevalence.
capped_imbalance | ResNet50 | fold 101 | rigorous | train=1369 val=362
  epoch 01/12 loss=0.6091 valAUC=0.8245 time=283s
  epoch 02/12 loss=0.4620 valAUC=0.8795 time=300s
  epoch 03/12 loss=0.4116 valAUC=0.8300 time=302s
  epoch 04/12 loss=0.3470 valAUC=0.8660 time=304s
  epoch 05/12 loss=0.3154 valAUC=0.8724 time=304s
  early stop
capped_imbalance | ResNet50 | fold 101 | controlled_leak | train=1369 val=362
  epoch 01/12 loss=0.5865 valAUC=0.8390 time=294s
  epoch 02/12 loss=0.4415 valAUC=0.8408 time=304s
  epoch 03/12 loss=0.3808 valAUC=0.8653 time=311s
  epoch 04/12 loss=0.3273 valAUC=0.8508 time=304s
  epoch 05/12 loss=0.2665 valAUC=0.8660 time=306s
  epoch 06/12 loss=0.2505 valAUC=0.8697 time=316s
  epoch 07/12 loss=0.2167 valAUC=0.8565 time=312s
  epoch 08/12 loss=0.181

In [15]:
# =========================
# 15. Final integrity summary + manuscript tables + reproducibility archive
# =========================
if imbalance_training:
    training_log = pd.concat([
        training_log,
        pd.DataFrame(imbalance_training),
    ], ignore_index=True)
    training_log.to_csv(
        OUT / "training_summary.csv",
        index=False,
    )

summary = {
    "dataset": "SIIM-ISIC 2020 training data",
    "standalone_run": True,
    "old_predictions_reused": False,
    "old_checkpoints_reused": False,
    "old_xai_reused": False,
    "primary_cohort_images": int(len(cohort)),
    "primary_cohort_patients": int(cohort.Patient_ID.nunique()),
    "primary_positive_prevalence": float(cohort.Label.mean()),
    "lesion_id_available": bool(LESION_ID_AVAILABLE),
    "lesion_id_coverage_pct": float(100*cohort.Lesion_ID.notna().mean()),
    "folds": FOLDS,
    "architectures": RUN_ARCHITECTURES,
    "decoded_content_duplicates_remaining": int(
        cohort.duplicated("decoded_fp").sum()
    ),
    "all_rigorous_patient_overlap_zero": bool(
        audit_df.rig_train_test_patient_overlap.eq(0).all()
    ),
    "all_rawhash_overlap_zero": bool(
        (
            audit_df.rig_train_test_rawhash_overlap.eq(0)
            & audit_df.leak_train_test_rawhash_overlap.eq(0)
        ).all()
    ),
    "all_decoded_overlap_zero": bool(
        (
            audit_df.rig_train_test_decoded_overlap.eq(0)
            & audit_df.leak_train_test_decoded_overlap.eq(0)
        ).all()
    ),
    "exact_target_diagnosis_site_sex_matching": bool(
        audit_df.exact_match_strata_preserved.all()
    ),
    "mean_test_sibling_exposure_pct": float(
        100*audit_df.test_exposure_pct.mean()
    ),
    "mean_candidate_utilization_pct": float(
        audit_df.candidate_utilization_pct.mean()
    ),
    "mean_age_smd_leak_minus_rig": float(
        audit_df.age_smd_leak_minus_rig.mean()
    ),
    "threshold_policy": (
        "Youden J selected from each final model's validation predictions, "
        "then locked before test evaluation"
    ),
    "deterministic_runtime": versions,
    "xai_architecture": XAI_ARCHITECTURE,
    "xai_method": (
        "true-class paired Grad-CAM cosine/Pearson similarity and centroid displacement"
    ),
    "xai_quantitative_images": int(len(xai_pairs)),
    "xai_visual_selection": (
        "seeded target/site/exposure-stratified selection before predictions"
    ),
    "capped_imbalance_sensitivity_run": bool(
        RUN_CAPPED_IMBALANCE_SENSITIVITY
    ),
}

with open(
    OUT / "final_integrity_summary.json",
    "w",
) as f:
    json.dump(summary, f, indent=2)

assert summary["decoded_content_duplicates_remaining"] == 0
assert summary["all_rigorous_patient_overlap_zero"]
assert summary["all_rawhash_overlap_zero"]
assert summary["all_decoded_overlap_zero"]
assert summary["exact_target_diagnosis_site_sex_matching"]
assert summary["old_predictions_reused"] is False

primary = stats[
    stats.Metric.isin([
        "AUC", "PR_AUC", "ACC",
        "Macro_F1", "Brier"
    ])
].copy()

primary.to_csv(
    OUT / "manuscript_primary_result_table.csv",
    index=False,
)

print("\n=== FINAL INTEGRITY SUMMARY ===")
print(json.dumps(summary, indent=2))

print("\n=== MANUSCRIPT-READY PRIMARY TABLE ===")
print(
    primary.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)

readme = f"""SIIM-ISIC 2020 FINAL FULL SAME-PATIENT LEAKAGE AUDIT

Standalone:
- previous notebook output used: NO
- old predictions/checkpoints/XAI reused: NO

Primary cohort:
- images: {len(cohort)}
- patients: {cohort.Patient_ID.nunique()}
- positive prevalence: {cohort.Label.mean():.6f}
- lesion_id available: {LESION_ID_AVAILABLE}
- lesion_id coverage: {100*cohort.Lesion_ID.notna().mean():.2f}%
- folds: {FOLDS}
- architectures: {", ".join(RUN_ARCHITECTURES)}

Final control:
- fixed identical test images for rigorous and controlled-leak arms;
- exact and decoded-pixel test content forbidden from both arms;
- rigorous training has zero test-patient overlap;
- controlled-leak replacements preserve target + diagnosis + site + sex exactly;
- nearest-age replacement within the exact matching stratum;
- separate patient-disjoint validation;
- validation-locked Youden-J threshold for thresholded primary metrics;
- fixed-0.5 metrics exported separately;
- patient-cluster bootstrap CIs + paired randomization;
- same-lesion / different-lesion exposure analysis when lesion_id is available;
- corrected ResNet-50 Grad-CAM attribution-stability analysis.

Key files:
- primary_cohort_manifest.csv
- primary_final_leakage_audit_by_fold.csv
- primary_oof_paired_predictions.csv
- primary_cluster_aware_metrics.csv
- supplementary_fixed_0p5_threshold_metrics.csv
- primary_exposure_groups.csv
- primary_exposure_trends.csv
- manuscript_primary_result_table.csv
- training_summary.csv
- final_integrity_summary.json
- splits/
- predictions/
- xai/preselected_quantitative_xai_cases.csv
- xai/preselected_visual_cases.csv
- xai/gradcam_paired_map_metrics.csv
- xai/gradcam_cluster_summary.csv

Capped imbalance sensitivity:
- enabled: {RUN_CAPPED_IMBALANCE_SENSITIVITY}
- benign cap: {IMBALANCE_MAX_BENIGN}
- explicitly NOT described as native/natural ISIC prevalence.
"""

(OUT / "README.txt").write_text(readme)

archive = BASE / "ISIC_Final_Full_Results.zip"

with zipfile.ZipFile(
    archive,
    "w",
    zipfile.ZIP_DEFLATED,
) as z:
    for p in OUT.rglob("*"):
        if p.is_file():
            z.write(
                p,
                p.relative_to(BASE),
            )

print("\nResults archive:", archive)
print("ISIC FINAL FULL ONE-RUN COMPLETE.")


=== FINAL INTEGRITY SUMMARY ===
{
  "dataset": "SIIM-ISIC 2020 training data",
  "standalone_run": true,
  "old_predictions_reused": false,
  "old_checkpoints_reused": false,
  "old_xai_reused": false,
  "primary_cohort_images": 1162,
  "primary_cohort_patients": 796,
  "primary_positive_prevalence": 0.5,
  "lesion_id_available": true,
  "lesion_id_coverage_pct": 100.0,
  "folds": 5,
  "architectures": [
    "ResNet50",
    "ConvNeXtTiny"
  ],
  "decoded_content_duplicates_remaining": 0,
  "all_rigorous_patient_overlap_zero": true,
  "all_rawhash_overlap_zero": true,
  "all_decoded_overlap_zero": true,
  "exact_target_diagnosis_site_sex_matching": true,
  "mean_test_sibling_exposure_pct": 45.5216812194761,
  "mean_candidate_utilization_pct": 98.75832238094571,
  "mean_age_smd_leak_minus_rig": -0.005525877243576895,
  "threshold_policy": "Youden J selected from each final model's validation predictions, then locked before test evaluation",
  "deterministic_runtime": {
    "python": "3.